In [13]:
from importlib.metadata import version

print("torch version:", version("torch"))

torch version: 2.8.0


In [ ]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

d:\Self study\Deep Learning\LLM from Scratch\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:279: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [14]:
input_query =  inputs[1]
input_query

tensor([0.5500, 0.8700, 0.6600])

#### Attention Scores

In [19]:
# empty tensors
att_scores = torch.empty(inputs.shape[0])

for i , emb in enumerate(inputs):
    att_scores[i] = torch.dot(emb , input_query)


att_scores
    

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])

#### Attention Weights

In [22]:
# get the normalization using softmax
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum()    

softmax_naive(att_scores)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [24]:
att_normalized =  torch.softmax(att_scores , dim=0)

In [29]:
# context vector = sum(attention_weights * embeddings)
ctx_vector = torch.sum(att_normalized.unsqueeze(1) * inputs, dim=0)


In [30]:
ctx_vector

tensor([0.4419, 0.6515, 0.5683])

Computing attention weights for all input tokens

In [37]:
inputs.shape

torch.Size([6, 3])

In [36]:
# empty tensors
att_scores = torch.empty(6,6)

for i , x_i in enumerate(inputs):
    for j , x_j in enumerate(inputs):
        att_scores[i,j] = torch.dot(x_i , x_j)


att_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [34]:
att_scores = inputs @ inputs.T
att_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [39]:
atn_weights = torch.softmax(att_scores , dim =1)
atn_weights

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [42]:
all_ctx_vecs =  atn_weights @ inputs
all_ctx_vecs

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])